# 03 - Adaptive retrieval depth: baselines and the proposed methods
| Family | Method | Idea |
|---|---|---|
| fixed | top-k | same k for every query |
| rule | relative score / z-score | keep passages above a threshold |
| prior art | **Adaptive-k** (Taguchi et al., 2025) | cut at the largest gap in the sorted similarity scores + buffer |
| prior art | **CAR-lite** (after Xu et al., 2025) | 2-means on scores, cut at cluster boundary |
| **proposed** | **RCAK** - Risk-Controlled Adaptive-k | gradient-boosted *quantile* regression of log(rank of first gold passage) from score-distribution features, then **split-conformal calibration**. Guarantees P(gold rank <= k_hat) >= 1 - alpha under exchangeability; alpha is an interpretable dial. |
| **proposed** | **CGC** - Confidence-Gated Cascade | classifier predicts P(gold in top-3); serve 3 if confident else 20 |
| bound | Oracle | smallest k containing the gold passage |

Features use **only retrieval-score statistics, dense/BM25 agreement and the question wh-word** - no LLM call, so the controller costs milliseconds.

**Outputs:** `csv/strategy_results.csv`, `csv/feature_matrix_sample.csv`

In [ ]:
# ---- Environment setup (works in Colab AND in a local venv / Jupyter) ----
import os, sys, pathlib, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "sentence-transformers", "python-docx", "psutil"], check=False)
    if not pathlib.Path("adaptive_rag").exists():
        from google.colab import files
        print("Upload rag_adaptive_project.zip (the project archive you downloaded)")
        up = files.upload(); z = [k for k in up if k.endswith(".zip")][0]
        zipfile.ZipFile(z).extractall("."); 
        root = [p for p in pathlib.Path(".").iterdir() if (p / "adaptive_rag").exists()][0]; os.chdir(root)
    try:
        from google.colab import drive; drive.mount("/content/drive")      # results persist on your Drive
        os.environ["RAG_RESULTS_DIR"] = "/content/drive/MyDrive/rag_adaptive_results"
    except Exception as e:
        os.environ["RAG_RESULTS_DIR"] = os.path.abspath("results")
    BACKEND = "st"          # real neural embeddings on Colab (GPU recommended: Runtime > Change runtime type > T4)
else:
    here = pathlib.Path.cwd()
    root = here.parent if here.name == "notebooks" else here
    sys.path.insert(0, str(root)); os.chdir(root)
    os.environ.setdefault("RAG_RESULTS_DIR", str(root / "results_notebooks"))
    BACKEND = "lsa"         # offline CPU baseline (TF-IDF+SVD). Set to "st" if sentence-transformers can download models.
sys.path.insert(0, os.getcwd())
CFG = dict(backend=BACKEND, st_model="sentence-transformers/all-MiniLM-L6-v2", n_queries=int(os.environ.get("RAG_NQ", 10000)), seed=0)
print("backend:", BACKEND, "| results ->", os.environ["RAG_RESULTS_DIR"])


In [ ]:
import numpy as np, pandas as pd
from adaptive_rag import experiment as E
from adaptive_rag.features import feature_groups
from adaptive_rag.utils import results_dir
B = E.prepare_cached(CFG)
B.feat.assign(split=B.split).head(200).to_csv(results_dir("csv")/"feature_matrix_sample.csv", index=False)
feature_groups()

In [ ]:
df, perq, te = E.evaluate(B, return_per_query=True)
df.to_csv(results_dir("csv")/"strategy_results.csv", index=False)
cols = ["strategy","param","mean_k","hit_rate","answer_in_context","mean_ctx_tokens_est"]
display(df[df.serve_retriever=="hybrid_rrf"][cols].round(3).sort_values(["strategy","mean_k"]).reset_index(drop=True))

### Matched-budget view (headline table)
For each adaptive method we find the fixed top-k with the **same mean number of passages** (linear interpolation along the fixed-k curve) and report the difference.

In [ ]:
iso = E.isobudget(B, "hybrid_rrf", n_boot=500); iso.to_csv(results_dir("csv")/"isobudget_hybrid.csv", index=False)
best = iso[iso.metric=="answer_in_context"].sort_values("diff", ascending=False).head(12)
display(best[["strategy","param","mean_k","adaptive_value","fixed_value_at_same_mean_k","diff","ci95_lo","ci95_hi"]].round(3))